<p class="lead">The linear model is the workhorse of quantitative science and the ancestor of every method in Block D. It is also the only model in this course whose parameters can be read as statements about the world rather than as machinery for prediction — which is why it is simultaneously the most useful and the most abused tool a thesis student has. This unit builds ordinary least squares from the normal equations, reads a regression table line by line, and then spends most of its length on the diagnostics that decide whether the table may be believed: residual structure, leverage and influence, multicollinearity, and the chance correlation that appears whenever there are too many descriptors for too few compounds. The applied thread is a <strong>Hansch-type QSAR</strong> on a congeneric series of ROCK1 inhibitors, which fails honestly and instructively; the unit then adds logistic regression with calibration, the ridge/lasso/elastic-net family as an explicit bias–variance trade-off, formal model comparison, and blocking as the cure for a confounded comparison. It closes on the question that should be asked before any model is fitted: are we explaining, or are we predicting?</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad construye el modelo lineal desde las ecuaciones normales y con <code>lm()</code>, y enseña a leer una tabla de regresión: coeficientes, errores estándar, intervalos, R² y R² ajustado. Se tratan en profundidad los diagnósticos (los cuatro gráficos de <code>plot(fit)</code>, apalancamiento, distancia de Cook, multicolinealidad y VIF con <code>car</code>), las transformaciones, los predictores categóricos y las interacciones, y el ANOVA como caso particular del modelo lineal. Se reproduce un QSAR de tipo Hansch sobre una serie congenérica de inhibidores de ROCK1 y se controla la correlación por azar (Topliss-Costello) mediante aleatorización de la variable respuesta. Se ajusta una regresión logística (razones de momios, separación, calibración), se comparan cresta, lazo y red elástica con <code>glmnet</code> usando validación cruzada y estabilidad por bootstrap, y se comparan modelos con R² ajustado, AIC, BIC y pruebas F anidadas. Finalmente se usa el diseño pareado de la matriz de expresión para ilustrar el bloqueo y la confusión, y se distingue el modelado explicativo del predictivo. Esta es la edición en R de la unidad.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Derive and compute ordinary least squares from the normal equations, and interpret coefficients, standard errors, confidence intervals, R² and adjusted R².</li>
<li>Diagnose a fitted model with residual plots, leverage, Cook's distance and variance inflation factors, and say what each diagnostic can and cannot detect.</li>
<li>Use transformations, categorical predictors and interactions, and recognise ANOVA as a linear model.</li>
<li>Fit a Hansch-type QSAR on a congeneric series, state the series-selection rule, and control for chance correlation by response randomisation.</li>
<li>Fit and interpret logistic regression: likelihood, odds ratios, separation, and calibration of predicted probabilities.</li>
<li>Explain ridge, lasso and elastic net as bias–variance trade-offs, choose the penalty by cross-validation, and assess the stability of lasso selection by bootstrap.</li>
<li>Compare models with adjusted R², AIC, BIC and nested F or likelihood-ratio tests, and know what each comparison assumes.</li>
<li>Use a blocking factor to remove a confound, and distinguish explanatory from predictive modelling.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read linearly, running each chunk; then refit the central model after removing one descriptor and watch every other coefficient move — that single experiment teaches multicollinearity better than any paragraph. Everything runs on base R's <code>stats</code><sup class="c"><a href="#r1">[1]</a></sup> plus <code>data.table</code>, <code>ggplot2</code>, <code>car</code><sup class="c"><a href="#r34">[34]</a></sup> for VIFs and heteroscedasticity tests, <code>glmnet</code><sup class="c"><a href="#r35">[35]</a></sup> for the penalised fits, <code>Matrix</code> for the sparse fingerprints and <code>pROC</code> for the AUC; in a fresh Google Colab R runtime run once: <code>install.packages(c("data.table","ggplot2","car","glmnet","pROC","lme4","R.utils"))</code>. This unit assumes <a href="../U7_Estimation/index.html">Unit 7</a> (standard errors, intervals, bootstrap) and <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a> (tests, effect sizes, multiplicity).</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(car); library(glmnet); library(Matrix); library(pROC)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_bw(base_size = 9) +
          theme(panel.grid.minor = element_blank(),
                plot.title = element_text(face = "bold", size = 10)))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| car", as.character(packageVersion("car")),
    "| glmnet", as.character(packageVersion("glmnet")),
    "| pROC", as.character(packageVersion("pROC")), "\n")


In [ ]:
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
D     <- function(f) data_file(f)
rock  <- fread(D("rock1_chembl_curated.csv"))
exact <- rock[censored == FALSE]
DESC  <- c("MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotB",
           "AromaticRings", "RingCount", "HeavyAtoms", "FractionCSP3", "QED")
cnt <- sort(table(exact$scaffold_smiles), decreasing = TRUE)
big <- cnt[cnt >= 30]
cat("compounds with an exact potency value :", nrow(exact), "\n")
cat("distinct Bemis-Murcko scaffolds       :", uniqueN(exact$scaffold_smiles), "\n")
cat("scaffold families with >= 30 compounds:", length(big), "-> sizes",
    paste(as.integer(big), collapse = ", "), "\n\n")
print(round(t(sapply(exact[, ..DESC], function(v)
  c(mean = mean(v), sd = sd(v), min = min(v), max = max(v)))), 2))



Eleven interpretable descriptors, 2,764 compounds with a measured potency, 1,218 distinct scaffolds and five scaffold families large enough to model separately: 207, 59, 43, 38 and 33 compounds. Those five families carry the whole unit.

## 1. What a linear model says

A linear model asserts that the expected value of a response is a weighted sum of predictors:

\[ y_i = \beta_0 + \beta_1 x_{i1} + \dots + \beta_p x_{ip} + \varepsilon_i,\qquad \varepsilon_i \stackrel{\text{iid}}{\sim} \mathcal{N}(0,\sigma^2). \]

"Linear" refers to the **coefficients**, not to the predictors: \( y = \beta_0 + \beta_1 x + \beta_2 x^2 \) is a linear model, and so is a model in log x or in an interaction term \( x_1x_2 \). The assumptions that matter, in decreasing order of how much damage their violation does:

1. **Independence** of the errors. Violated by repeated measurements of the same patient, frames of the same trajectory, compounds from the same series. No amount of data fixes it; the model must be changed (blocking, §11).
2. **Correct functional form** — the mean really is a weighted sum of these predictors. Violated silently; residual plots are the detector (§4).
3. **Homoscedasticity** — constant error variance. Inflates or deflates standard errors when violated.
4. **Normality of the errors** — needed only for exact small-sample inference; irrelevant for large n thanks to the central limit theorem of <a href="../U7_Estimation/index.html">Unit 7</a>.

<div class="box bridge"><h4>Domain context — congeneric series and Hansch analysis</h4>
<p>A <strong>congeneric series</strong> is a family of molecules sharing one core structure (the <strong>scaffold</strong>) and differing only in the substituents hung off it. Medicinal chemists work this way: once a promising core is found, dozens of analogues are made by varying one position at a time. In 1962–1964 Corwin Hansch and Toshio Fujita proposed that, within such a series, biological potency can be written as a linear function of a few physicochemical properties of the substituents — lipophilicity (log P), an electronic term and a steric term<sup class="c"><a href="#r2">[2]</a></sup><sup class="c"><a href="#r3">[3]</a></sup>. That proposal created <strong>QSAR</strong> (quantitative structure–activity relationship) and it is, mathematically, nothing but multiple linear regression. Its classical practice rules — use a congeneric series, use at least five compounds per descriptor, avoid collinear descriptors, check for chance correlation<sup class="c"><a href="#r4">[4]</a></sup> — are exactly the linear-model diagnostics of this unit, discovered independently by chemists. The descriptors in our table are whole-molecule properties computed by RDKit during curation: molecular weight, calculated log P, topological polar surface area (TPSA, a proxy for membrane permeability), counts of hydrogen-bond donors and acceptors and of rotatable bonds, ring counts, the fraction of sp<sup>3</sup> carbons (a measure of three-dimensionality) and QED, a composite drug-likeness score between 0 and 1. The <strong>Bemis–Murcko scaffold</strong><sup class="c"><a href="#r5">[5]</a></sup> is the computable definition of "the core" used to group compounds into series, and the data come from ChEMBL<sup class="c"><a href="#r32">[32]</a></sup>, curated in Unit 6. The three-dimensional descendant of Hansch analysis, comparative molecular field analysis (CoMFA)<sup class="c"><a href="#r29">[29]</a></sup>, replaces the handful of descriptors with thousands of interaction-energy values on a grid and then regularises — the same bias–variance problem as §9, forty years earlier.</p></div>

## 2. Ordinary least squares from the normal equations

OLS chooses the coefficients that minimise the sum of squared residuals \( \lVert y - X\beta \rVert^2 \). Differentiating and setting the gradient to zero gives the **normal equations** \( X^{\mathsf T}X\hat\beta = X^{\mathsf T}y \), so

\[ \hat\beta = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y, \qquad \widehat{\operatorname{Var}}(\hat\beta) = \hat\sigma^{2}(X^{\mathsf T}X)^{-1},\qquad \hat\sigma^{2} = \frac{\lVert y - X\hat\beta\rVert^{2}}{n-p-1}. \]

### 2.1 Worked by hand

Take five compounds with calculated log P values 2.0, 3.0, 4.0, 5.0, 6.0 and potencies 6.0, 6.6, 7.0, 7.1, 7.3. With a single predictor the normal equations reduce to the familiar formulas.

Mean of x = (2+3+4+5+6)/5 = 4.0; mean of y = (6.0+6.6+7.0+7.1+7.3)/5 = 34.0/5 = 6.8.

| x | y | x − x̄ | y − ȳ | (x−x̄)(y−ȳ) | (x−x̄)² |
|---|---|---|---|---|---|
| 2.0 | 6.0 | −2 | −0.8 | 1.60 | 4 |
| 3.0 | 6.6 | −1 | −0.2 | 0.20 | 1 |
| 4.0 | 7.0 | 0 | +0.2 | 0.00 | 0 |
| 5.0 | 7.1 | +1 | +0.3 | 0.30 | 1 |
| 6.0 | 7.3 | +2 | +0.5 | 1.00 | 4 |
| | | | **Σ** | **3.10** | **10** |

Slope \( \hat\beta_1 = 3.10/10 = 0.310 \); intercept \( \hat\beta_0 = 6.8 - 0.310 \times 4.0 = 5.560 \).

Fitted values: 6.180, 6.490, 6.800, 7.110, 7.420. Residuals: −0.180, +0.110, +0.200, −0.010, −0.120. Residual sum of squares = 0.0324 + 0.0121 + 0.0400 + 0.0001 + 0.0144 = **0.0990**. Total sum of squares = 0.64 + 0.04 + 0.04 + 0.09 + 0.25 = **1.0600**, so \( R^2 = 1 - 0.0990/1.0600 = 0.9066 \).

Residual standard error \( \hat\sigma = \sqrt{0.0990/3} = 0.1817 \); the standard error of the slope is \( \hat\sigma/\sqrt{\sum(x-\bar x)^2} = 0.1817/\sqrt{10} = 0.0574 \), giving \( t = 0.310/0.0574 = 5.40 \) on 3 degrees of freedom.



In [ ]:
xv <- c(2, 3, 4, 5, 6); yv <- c(6.0, 6.6, 7.0, 7.1, 7.3)
b1 <- sum((xv - mean(xv)) * (yv - mean(yv))) / sum((xv - mean(xv))^2)
b0 <- mean(yv) - b1 * mean(xv)
Xm <- cbind(1, xv)
beta_ne <- solve(t(Xm) %*% Xm, t(Xm) %*% yv)       ## the normal equations, solved directly
fit0 <- lm(yv ~ xv); s0 <- summary(fit0)
resid0 <- yv - Xm %*% beta_ne
sigma2 <- sum(resid0^2) / (length(xv) - 2)
se_hand <- sqrt(diag(sigma2 * solve(t(Xm) %*% Xm)))
cat(sprintf("by hand          : b0 = %.4f, b1 = %.4f\n", b0, b1))
cat(sprintf("normal equations : b0 = %.4f, b1 = %.4f\n", beta_ne[1], beta_ne[2]))
cat(sprintf("lm()             : b0 = %.4f, b1 = %.4f\n", coef(fit0)[1], coef(fit0)[2]))
cat(sprintf("RSS %.4f | TSS %.4f | R2 %.4f (lm %.4f)\n", sum(resid0^2),
            sum((yv - mean(yv))^2), 1 - sum(resid0^2) / sum((yv - mean(yv))^2), s0$r.squared))
cat(sprintf("sigma_hat %.4f | SE(b1) by hand %.4f | lm %.4f | t = %.2f\n",
            sqrt(sigma2), se_hand[2], s0$coefficients[2, 2], s0$coefficients[2, 3]))
cat(sprintf("95%% CI for the slope: [%.4f, %.4f]\n", confint(fit0)[2, 1], confint(fit0)[2, 2]))



Every number from the hand calculation is reproduced to four decimals. This is worth doing once, because it makes the regression table stop being an oracle: the slope is a ratio of two sums, the standard error is a scaled diagonal element of an inverted matrix, and the t statistic is their quotient. Note that the interval on the slope, 0.127 to 0.493, is extremely wide — five points buy very little precision.

<div class="box warn"><h4>Never invert the matrix in production code</h4>
<p><code>solve(t(X) %*% X, t(X) %*% y)</code> is used above for transparency. <code>lm()</code> uses a QR decomposition of X, which is numerically far more stable when the columns are nearly collinear (§5), and it also detects and reports exactly collinear columns instead of silently producing nonsense. Computing <code>solve(t(X) %*% X)</code> on a descriptor matrix with correlated columns can lose most of the available precision.</p></div>

## 3. Reading a regression table

The next chunk fits the Hansch-type model on a real congeneric series. The **series-selection rule**, stated explicitly because it is a modelling decision a referee will ask about: among compounds with an exact (non-censored) measurement, group by Bemis–Murcko generic scaffold, keep families with at least 30 members, and take the largest. That family has 207 compounds for 11 descriptors — 18.8 compounds per descriptor, comfortably above the classical requirement of five<sup class="c"><a href="#r4">[4]</a></sup>.



In [ ]:
series <- exact[scaffold_smiles == names(cnt)[1]]
hansch <- lm(as.formula(paste("pActivity ~", paste(DESC, collapse = " + "))), data = series)
sh <- summary(hansch)
cat(sprintf("series size n = %d | descriptors p = %d | compounds per descriptor = %.1f\n",
            nrow(series), length(DESC), nrow(series) / length(DESC)))
cat(sprintf("R2 = %.3f | adjusted R2 = %.3f | F = %.2f on (%d, %d) df | p(F) = %.4f\n",
            sh$r.squared, sh$adj.r.squared, sh$fstatistic[1], sh$fstatistic[2], sh$fstatistic[3],
            pf(sh$fstatistic[1], sh$fstatistic[2], sh$fstatistic[3], lower.tail = FALSE)))
cat(sprintf("residual standard error = %.3f log units (target SD = %.3f)\n",
            sh$sigma, sd(series$pActivity)))
cat("coefficients dropped as aliased (NA):", paste(names(which(is.na(coef(hansch)))), collapse = ", "), "\n\n")
print(round(sh$coefficients, 4))



Read the table in this order, and in no other:

1. **The F test and R².** Here R² is 0.078 and the overall F test gives p = 0.059 on (9, 197) degrees of freedom. The eleven whole-molecule descriptors together explain less than a tenth of the potency variation inside this series, and the model as a whole is not significant. **This is the result, and it is a real one.**
2. **Only then, the coefficients.** Reading individual coefficients from a model that does not pass its own F test is the most common way to generate a false QSAR narrative. Two coefficients here have p-values near 0.05 (FractionCSP3 at 0.053, HBA at 0.079) — which is exactly what nine predictors produce on noise.
3. **The residual standard error**, 0.797 log units against a target standard deviation of 0.812 — the model has reduced the unexplained variation by under 2 %.

Two things R did that are worth noticing. The F test has **9** numerator degrees of freedom, not 11, and two coefficients are reported as `NA`: `AromaticRings` and `RingCount` are **constant within this series** and therefore perfectly collinear with the intercept. R detects the aliasing through its QR pivoting, drops those columns and tells you by printing NA — behaviour worth preferring to a library that silently returns a number. The constancy is not a bug but the definition of a congeneric series: every member has the same ring system.

The chemistry behind the failure is worth stating, because students usually assume the code is wrong. Inside an optimised congeneric series, molecules have nearly the same size, lipophilicity and polarity by construction; what differs is *where* a substituent sits and what specific contact it makes with the protein. Whole-molecule descriptors are blind to that. Classical Hansch analysis used substituent constants (Hammett σ, Hansch π, Taft E<sub>s</sub>) measured per substituent position, not whole-molecule properties, which is precisely the difference. The lesson for the thesis: **a QSAR that fails to fit is not a failure of the student; it is information about the descriptors.**

The residual standard error also deserves a comparison the field rarely makes: Unit 7 estimated the experimental noise floor of this dataset at 0.406 log units, and independent analyses of public bioactivity data reach similar figures<sup class="c"><a href="#r33">[33]</a></sup>. A model whose residual error is 0.797 log units is therefore about twice as far from the labels as the labels are from each other — there is room to improve, but not unlimited room<sup class="c"><a href="#r30">[30]</a></sup>.



In [ ]:
y <- series$pActivity
pred <- fitted(hansch)
r2_grow <- numeric(61); r2_adj <- numeric(61); Xg <- matrix(1, nrow = length(y))
for (k in 0:60) {
  if (k > 0) { set.seed(k); Xg <- cbind(Xg, rnorm(length(y))) }
  ff  <- lm.fit(Xg, y)
  rss <- sum(ff$residuals^2); tss <- sum((y - mean(y))^2)
  r2_grow[k + 1] <- 1 - rss / tss
  r2_adj[k + 1]  <- 1 - (1 - r2_grow[k + 1]) * (length(y) - 1) / (length(y) - k - 1)
}
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1), cex = 0.75)
lims <- range(c(pred, y)) + c(-0.2, 0.2)
plot(pred, y, pch = 20, cex = 0.6, col = PAL["blue"], xlim = lims, ylim = lims,
     xlab = "predicted pActivity", ylab = "observed pActivity",
     main = sprintf("observed vs predicted (R2 = %.3f)", sh$r.squared))
abline(0, 1, lty = 2)
plot(0:60, r2_grow, type = "l", col = PAL["orange"], lwd = 1.6, ylim = c(-0.1, 0.4),
     xlab = "number of PURE NOISE predictors added", ylab = "value",
     main = "R2 cannot decrease; adjusted R2 can")
lines(0:60, r2_adj, col = PAL["blue"], lwd = 1.6); abline(h = 0, lwd = 0.8)
legend("topleft", c("R2", "adjusted R2"), col = unname(PAL[c("orange", "blue")]), lwd = 1.6, bty = "n", cex = 0.85)
par(op)
cat(sprintf("observed range %.2f - %.2f, predicted range %.2f - %.2f  <- the model barely varies\n",
            min(y), max(y), min(pred), max(pred)))
cat(sprintf("with 60 random predictors on %d compounds: R2 = %.3f but adjusted R2 = %.3f\n",
            length(y), r2_grow[61], r2_adj[61]))
cat(sprintf("the real model with 9 estimable descriptors: R2 = %.3f, adjusted R2 = %.3f\n",
            sh$r.squared, sh$adj.r.squared))



<p class="caption"><b>Figure 1.</b> Left: observed against predicted potency for the Hansch model. The predictions span 6.76 to 7.86 — barely one log unit — while the observations span 5.39 to 9.70; a model with almost no explanatory power produces a nearly vertical cloud, which is the visual signature to recognise. Right: the same 207 compounds regressed on an increasing number of columns of <em>pure random noise</em>. R² (orange) rises monotonically and reaches 0.267 with 60 noise predictors; adjusted R² (blue) ends at −0.034, which is exactly its job. This is why R² alone is never evidence, and it is the mechanism behind the chance correlation of §7.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Refit the same model on the whole exact subset (2,764 compounds, all scaffolds) instead of one series: <code>lm(as.formula(paste("pActivity ~", paste(DESC, collapse = "+"))), data = exact)</code>. R² rises and several coefficients become strongly significant. Explain why a model can fit better across series than within one, and why the across-series coefficients are nevertheless harder to interpret causally (hint: what varies between series that does not vary within one?).</p></div>

## 4. Diagnostics: four plots that must be looked at

A regression table cannot reveal curvature, heteroscedasticity, or a single compound driving the whole fit. Four plots can, and in R they are the default `plot()` method of a fitted model — one of the strongest arguments for fitting with `lm()` rather than by hand.



In [ ]:
op <- par(mfrow = c(2, 2), mar = c(4, 4, 2.6, 1.2), cex = 0.7)
plot(hansch, which = 1:4, col = PAL["blue"], pch = 20, cex = 0.6)
par(op)
usable <- DESC[sapply(DESC, function(cc) uniqueN(series[[cc]])) > 1]
p1 <- length(usable) + 1
hat <- hatvalues(hansch); ck <- cooks.distance(hansch)
cat(sprintf("average leverage (p+1)/n = %.4f ; rule-of-thumb flag at 2x = %.4f ; max observed = %.4f\n",
            p1 / nrow(series), 2 * p1 / nrow(series), max(hat)))
cat(sprintf("compounds above the leverage flag: %d\n", sum(hat > 2 * p1 / nrow(series))))
cat(sprintf("largest Cook's distances: %s  (flag at 4/n = %.3f)\n",
            paste(sprintf("%.3f", sort(ck, decreasing = TRUE)[1:3]), collapse = ", "), 4 / nrow(series)))
cat(sprintf("Breusch-Pagan (car::ncvTest) p = %.4f\n", ncvTest(hansch)$p))
cat(sprintf("Shapiro-Wilk on the residuals p = %.4f\n", shapiro.test(residuals(hansch))$p.value))



<p class="caption"><b>Figure 2.</b> The four diagnostics of the Hansch model as R draws them. Top left: residuals against fitted values with a loess line — flat, so there is no gross curvature to fix with a transformation. Top right: the standardised residuals follow the normal line in the middle and depart in both tails; Shapiro–Wilk rejects normality at p = 0.0046, which with 207 points means a real but mild departure of the kind potency data usually show — not a reason to abandon the model, but a reason to prefer a bootstrap interval over the tabulated one if a coefficient mattered. Bottom left: the scale–location plot shows no obvious funnel, although the Breusch–Pagan test is significant (p = 0.022), so the standard errors in the table are mildly optimistic. Bottom right: Cook's distance per observation; the three largest are 0.087, 0.059 and 0.046, four to five times the conventional 4/n = 0.019 alarm, and 16 compounds exceed the 2(p+1)/n leverage flag. The fit — weak as it is — leans on a handful of unusual molecules, and any interpretation would have to be checked with and without them.</p>

The four plots answer four different questions, and only together do they constitute a check:

- **Residuals vs fitted** detects a wrong functional form — a U shape means a squared term or a log transform is missing.
- **QQ of residuals** checks the normality used by the small-sample inference. Heavy tails suggest robust regression or a bootstrap interval.
- **Scale–location** detects heteroscedasticity, which makes standard errors wrong (usually too small) without biasing the coefficients.
- **Cook's distance / leverage** finds influential points. **Leverage** (`hatvalues`) measures how unusual a compound's *descriptors* are, independently of its potency; **Cook's distance**<sup class="c"><a href="#r6">[6]</a></sup> combines leverage with residual size to measure how much the fitted model would move if that compound were deleted. `plot(fit, which = 5)` draws the two against each other, and `influence.measures(fit)` tabulates every diagnostic at once.

<div class="box warn"><h4>High leverage is not a reason to delete a compound</h4>
<p>A high-leverage compound is an unusual molecule; it may be the most informative one in the series. Delete a point only for a documented reason (a curation error, a different assay, an off-scaffold structure), report that you did, and show the fit with and without it. Deleting points because they spoil the fit is p-hacking with extra steps (<a href="../U8_Hypothesis_Testing/index.html">Unit 8</a>, §7).</p></div>

## 5. Multicollinearity and the variance inflation factor

When predictors are correlated, \( X^{\mathsf T}X \) is close to singular and \( (X^{\mathsf T}X)^{-1} \) has large entries: the coefficients become unstable and their standard errors explode, even though the model's *predictions* may be perfectly good. The diagnostic is the **variance inflation factor**, \( \mathrm{VIF}_j = 1/(1-R_j^2) \), where \( R_j^2 \) is the R² of regressing descriptor \( j \) on all the others. A VIF of 10 means the standard error of that coefficient is √10 ≈ 3.2 times larger than it would be with uncorrelated descriptors<sup class="c"><a href="#r7">[7]</a></sup><sup class="c"><a href="#r8">[8]</a></sup>. In R it is `car::vif()`, which refuses to run on a model with aliased coefficients — so the constant descriptors must be removed first, by hand.



In [ ]:
cat("descriptors that are CONSTANT inside this series (zero information):",
    paste(setdiff(DESC, usable), collapse = ", "), "\n")
cat(sprintf("-> keeping %d descriptors\n\n", length(usable)))
fit_us <- lm(as.formula(paste("pActivity ~", paste(usable, collapse = " + "))), data = series)
vifs <- sort(vif(fit_us), decreasing = TRUE)
cm <- cor(as.matrix(series[, ..usable]))
op <- par(mfrow = c(1, 2), mar = c(6, 6, 2.6, 1), cex = 0.7)
image(1:length(usable), 1:length(usable), cm[, length(usable):1], zlim = c(-1, 1),
      col = hcl.colors(21, "RdBu", rev = TRUE), axes = FALSE, xlab = "", ylab = "",
      main = "descriptor correlation")
axis(1, 1:length(usable), usable, las = 2, cex.axis = 0.75)
axis(2, 1:length(usable), rev(usable), las = 2, cex.axis = 0.75)
bp <- barplot(rev(vifs), horiz = TRUE, las = 1, col = PAL["blue"], xlab = "variance inflation factor",
              main = "VIF (flags at 5 and 10)", cex.names = 0.75)
abline(v = c(5, 10), col = c(PAL["orange"], "red"), lty = 2)
par(op)
print(round(vifs, 1))
diag(cm) <- 0
cat(sprintf("\nlargest absolute off-diagonal correlation: %.3f\n", max(abs(cm))))



<p class="caption"><b>Figure 3.</b> Left: the descriptor correlation matrix inside the series, showing the expected blocks — molecular weight, heavy-atom count and the surface-area terms move together (largest absolute correlation 0.911), as do the hydrogen-bonding counts. Right: variance inflation factors, with the conventional flags at 5 and 10. Five of the nine estimable descriptors sit above 9 and QED reaches 24.9, because QED is by construction a function of the others — a desirability score computed from molecular weight, log P, TPSA, donors, acceptors and rotatable bonds — while heavy-atom count (15.1) and molecular weight (13.9) are nearly the same quantity within this series. None of those coefficients may be interpreted individually.</p>

The experiment that makes collinearity tangible is to drop one descriptor and watch the others move.



In [ ]:
worst <- names(vifs)[1]
keep_cols <- setdiff(usable, worst)
fit_drop <- lm(as.formula(paste("pActivity ~", paste(keep_cols, collapse = " + "))), data = series)
ca <- coef(summary(fit_us)); cb <- coef(summary(fit_drop))
common <- intersect(rownames(ca), rownames(cb))
cmp_coef <- data.table(term = common,
                       coef_with = ca[common, 1], coef_without = cb[common, 1],
                       se_with = ca[common, 2], se_without = cb[common, 2])
cmp_coef[, change := coef_without - coef_with]
cat(sprintf("dropped descriptor: %s (VIF %.1f)\n", worst, vifs[1]))
cat(sprintf("R2 with %.4f -> without %.4f   (the fit barely changes)\n",
            summary(fit_us)$r.squared, summary(fit_drop)$r.squared))
cat(sprintf("largest absolute coefficient change: %.3f on %s\n",
            max(abs(cmp_coef$change)), cmp_coef$term[which.max(abs(cmp_coef$change))]))
cat(sprintf("mean reduction in standard error: %.1f %%\n",
            100 * (1 - mean(cmp_coef$se_without / cmp_coef$se_with))))
knitr::kable(cmp_coef, digits = 4)



Removing QED leaves the model's fit essentially unchanged — R² moves from 0.0785 to 0.0724 — while individual coefficients shift substantially (the intercept by 6.4 units) and every standard error shrinks, by 17 % on average. That is the entire phenomenon: with collinear predictors the *sum* of their contributions is well determined and the *split* between them is not. A coefficient that changes sign when a correlated descriptor is removed should never be given a chemical interpretation.

Three responses to multicollinearity, in order of preference: (1) drop or merge redundant descriptors on chemical grounds; (2) accept it, if prediction rather than interpretation is the goal — collinearity does not hurt predictions; (3) regularise (§9), which trades a little bias for a large reduction in coefficient variance.

## 6. Categorical predictors, interactions, and ANOVA as a linear model

A categorical predictor with k levels enters the model as k − 1 indicator ("dummy") columns, so its coefficients are differences from a reference level — in R, the first level of the factor, which `relevel()` changes. An **interaction** term \( x_1 \cdot x_2 \), written `x1 * x2` in a formula, lets the slope on one predictor depend on the value of another. And a one-way ANOVA is nothing but a linear model whose only predictor is a categorical variable: the F test of the ANOVA is the F test of that model<sup class="c"><a href="#r9">[9]</a></sup>.



In [ ]:
top5 <- names(cnt)[1:5]
sub <- exact[scaffold_smiles %in% top5]
sub[, series := factor(paste0("S", match(scaffold_smiles, top5)))]
lm_cat <- lm(pActivity ~ series, data = sub)
print(round(coef(summary(lm_cat)), 4))
cat("\nANOVA table of the same model:\n")
print(anova(lm_cat))
cat(sprintf("\noneway.test(var.equal = TRUE) on the same data: F = %.4f  <- identical\n",
            oneway.test(pActivity ~ series, data = sub, var.equal = TRUE)$statistic))
lm_add <- lm(pActivity ~ MolLogP + series, data = sub)
lm_int <- lm(pActivity ~ MolLogP * series, data = sub)
cat(sprintf("\nadditive model   : R2 %.3f, AIC %.1f\n", summary(lm_add)$r.squared, AIC(lm_add)))
cat(sprintf("interaction model: R2 %.3f, AIC %.1f\n", summary(lm_int)$r.squared, AIC(lm_int)))
print(anova(lm_add, lm_int))



The F statistic from `anova()` on the linear model and the one from `oneway.test()` are the same number to four decimals (31.8669), because they are the same test. Seeing this once removes the idea that ANOVA and regression are different techniques: ANOVA is the special case in which all predictors are categorical, and the familiar "sum of squares between groups over sum of squares within groups" is the model's explained variance over its residual variance. The coefficient table shows the other half of the equivalence: each `seriesSk` coefficient is the difference in mean pActivity between series k and the reference series S1.

The interaction model asks whether the slope on lipophilicity differs between series. `anova(lm_add, lm_int)` performs the nested F test — the exact small-sample version of the likelihood-ratio test — and returns F = 0.246 on (4, 370) degrees of freedom with p = 0.912: the four extra slope parameters buy 0.66 of residual sum of squares out of 249. AIC agrees, rising from 931.7 to 938.7, so the simpler additive model is preferred.



In [ ]:
slopes <- sapply(levels(sub$series), function(l) coef(lm(pActivity ~ MolLogP, data = sub[series == l]))[2])
lab <- sprintf("%s (n=%d, slope %+.2f)", levels(sub$series),
               sub[, .N, by = series][order(series)]$N, slopes)
p <- ggplot(sub, aes(MolLogP, pActivity, colour = series)) +
  geom_point(size = 0.6, alpha = 0.5) +
  geom_smooth(method = "lm", se = FALSE, linewidth = 0.7, formula = y ~ x) +
  scale_colour_manual(values = unname(c(PAL, "#777777")), labels = lab, name = NULL) +
  labs(title = "Separate slopes per series: what an interaction term estimates")
print(p)
cat(sprintf("pooled slope (additive model): %+.3f\n", coef(lm_add)["MolLogP"]))
for (l in levels(sub$series))
  cat(sprintf("  %s: %+.3f\n", l, slopes[l]))



<p class="caption"><b>Figure 4.</b> Potency against lipophilicity within each of the five series, with a separate least-squares line per series. The slopes range from −0.31 (S3) to +0.02 (S4) around a pooled slope of −0.179, and the lines have visibly different intercepts, which is what an interaction term formalises; but the scatter around each line is large and the per-series sample sizes are small (33 to 207), so the nested F test above cannot distinguish these five slopes from a single common one. The figure is the right way to decide whether an interaction is worth testing — look first, then test, and say in the paper that you looked.</p>

### 6.1 Transformations and the parabolic Hansch model

Hansch's own models were rarely linear in log P: potency usually rises with lipophilicity, peaks, and falls again as the compound becomes too greasy to stay in solution or too promiscuous to be selective. He captured that with a **quadratic term**, and the resulting parabolic model is the most-cited functional form in classical QSAR. Adding a squared term keeps the model linear in its coefficients, so nothing about the estimation changes — only the design matrix.



In [ ]:
lin  <- lm(pActivity ~ MolLogP, data = exact)
quad <- lm(pActivity ~ MolLogP + I(MolLogP^2), data = exact)
xs <- seq(quantile(exact$MolLogP, 0.01), quantile(exact$MolLogP, 0.99), length.out = 100)
pq <- predict(quad, newdata = data.frame(MolLogP = xs), interval = "confidence")
binned <- exact[, .(m = mean(pActivity), mid = mean(MolLogP)),
                by = .(bin = cut(MolLogP, 12))]
p <- ggplot(exact, aes(MolLogP, pActivity)) +
  geom_point(size = 0.25, alpha = 0.2, colour = "grey50") +
  geom_ribbon(data = data.frame(MolLogP = xs, lo = pq[, "lwr"], hi = pq[, "upr"]),
              aes(y = NULL, ymin = lo, ymax = hi), fill = PAL["orange"], alpha = 0.3) +
  geom_line(data = data.frame(MolLogP = xs, pActivity = pq[, "fit"]), colour = PAL["orange"], linewidth = 0.8) +
  geom_abline(intercept = coef(lin)[1], slope = coef(lin)[2], colour = PAL["blue"], linewidth = 0.8) +
  geom_point(data = binned, aes(mid, m), shape = 22, fill = "black", size = 1.6) +
  coord_cartesian(xlim = range(xs)) +
  labs(title = "Is the lipophilicity-potency relationship curved?")
print(p)
cat(sprintf("linear    : R2 = %.4f, AIC = %.1f\n", summary(lin)$r.squared, AIC(lin)))
cat(sprintf("parabolic : R2 = %.4f, AIC = %.1f, quadratic coefficient %+.4f (p = %.3g)\n",
            summary(quad)$r.squared, AIC(quad), coef(quad)[3], summary(quad)$coefficients[3, 4]))
print(anova(lin, quad))
cat(sprintf("likelihood-ratio chi2 = %.2f on 1 df, p = %.3g\n",
            2 * as.numeric(logLik(quad) - logLik(lin)),
            pchisq(2 * as.numeric(logLik(quad) - logLik(lin)), 1, lower.tail = FALSE)))
cat(sprintf("implied optimum log P = %.2f  (data range %.1f to %.1f)\n",
            -coef(quad)[2] / (2 * coef(quad)[3]), min(exact$MolLogP), max(exact$MolLogP)))



<p class="caption"><b>Figure 5.</b> Potency against calculated lipophilicity for all 2,764 compounds, with the linear fit (blue), the parabolic fit with its 95 % confidence band (orange) and twelve binned means (black squares) that show the shape of the data independently of either model. The quadratic term is strongly supported — the nested F test gives F = 55.9 on (1, 2761) df, p = 1.0 × 10<sup>−13</sup>, and the likelihood-ratio chi-square is 55.4 — and the fitted parabola peaks at log P 3.30, a chemically plausible place for an optimum. And yet the parabolic model explains 2.05 % of the variance against the linear model's 0.06 %: the curvature is real but tiny. An "optimum log P" quoted from a model with R² of 0.02 is a number a referee will not accept, and reporting the R² next to it is what keeps the claim honest.</p>

## 7. Chance correlation: the Topliss–Costello experiment

In 1972 Topliss and Costello showed by simulation that if you search enough descriptors on few enough compounds, an impressive correlation appears from random numbers<sup class="c"><a href="#r4">[4]</a></sup>. The expected R² of a regression of pure noise on p predictors with n observations is approximately p/(n−1) — with 9 estimable descriptors and 33 compounds, that is 0.281 *before any real signal exists*.

Our fifth series has 33 compounds. Fitting the same descriptors to it gives a respectable-looking R². The experiment that decides whether to believe it is **response randomisation** (y-scrambling): permute the potencies, refit, repeat, and look at where the real R² falls in the resulting null distribution.



In [ ]:
small <- exact[scaffold_smiles == names(cnt)[5]]
us_small <- DESC[sapply(DESC, function(cc) uniqueN(small[[cc]])) > 1]
fit_small <- lm(as.formula(paste("pActivity ~", paste(us_small, collapse = " + "))), data = small)
Xsm <- model.matrix(fit_small)[, -1]
set.seed(0)
scram <- replicate(2000, summary(lm(sample(small$pActivity) ~ Xsm))$r.squared)
r2_real <- summary(fit_small)$r.squared
p <- ggplot(data.frame(r2 = scram), aes(r2)) +
  geom_histogram(bins = 45, fill = "grey75", colour = NA) +
  geom_vline(xintercept = r2_real, colour = PAL["orange"], linewidth = 1) +
  geom_vline(xintercept = mean(scram), linetype = 2) +
  labs(x = "R2", y = "count",
       title = sprintf("Chance correlation: %d descriptors, %d compounds", length(us_small), nrow(small)))
print(p)
cat(sprintf("series size n = %d, estimable descriptors p = %d -> %.1f compounds per descriptor\n",
            nrow(small), length(us_small), nrow(small) / length(us_small)))
cat(sprintf("fit on the real data : R2 = %.3f, adjusted R2 = %.3f, F p-value = %.4f\n",
            r2_real, summary(fit_small)$adj.r.squared,
            pf(summary(fit_small)$fstatistic[1], summary(fit_small)$fstatistic[2],
               summary(fit_small)$fstatistic[3], lower.tail = FALSE)))
cat(sprintf("null from scrambling : mean R2 = %.3f, 95th percentile = %.3f, maximum = %.3f\n",
            mean(scram), quantile(scram, 0.95), max(scram)))
cat(sprintf("expected under noise, p/(n-1) = %.3f\n", length(us_small) / (nrow(small) - 1)))
cat(sprintf("fraction of scrambles reaching the real R2: %.4f\n", mean(scram >= r2_real)))



<p class="caption"><b>Figure 6.</b> The chance-correlation experiment (orange line: the real fit; dashed line: the mean of the null). Scrambling the potencies 2,000 times and refitting the same nine descriptors on the same 33 compounds produces R² values averaging 0.281 — exactly the theoretical p/(n−1) = 0.281 — with a 95th percentile of 0.493 and a maximum of 0.681. The real fit, R² = 0.595, is beyond the 95th percentile (0.65 % of scrambles do better) but is manifestly <em>not</em> the "R² = 0.6, excellent model" of a naive report: nearly half of that R² would have appeared from random numbers.</p>

Two numbers in that output are the antidote to the naive report. The **adjusted R²**, which penalises the number of predictors,

\[ R^2_{\text{adj}} = 1 - (1-R^2)\frac{n-1}{n-p-1}, \]

falls from 0.595 to 0.436 — the honest figure. And the empirical scrambling p-value of 0.0065 is the one to quote. Every QSAR paper from this laboratory reports a y-randomisation control for exactly this reason<sup class="c"><a href="#r10">[10]</a></sup><sup class="c"><a href="#r11">[11]</a></sup><sup class="c"><a href="#r12">[12]</a></sup>, and Unit 12 repeats the experiment for machine-learning models, where the number of effective parameters is much larger and the danger correspondingly worse<sup class="c"><a href="#r13">[13]</a></sup>.

## 8. Logistic regression

When the response is binary — active or inactive, responder or non-responder — the linear model is applied to the **log-odds** instead of to the response:

\[ \log\frac{\Pr(y=1\mid x)}{1-\Pr(y=1\mid x)} = \beta_0 + \beta_1x_1 + \dots + \beta_px_p . \]

The coefficients are fitted by maximising the Bernoulli likelihood rather than by least squares — `glm(..., family = binomial)` in R — and each \( e^{\beta_j} \) is an **odds ratio**: the multiplicative change in the odds of being active for a one-unit increase in \( x_j \), holding the others fixed. With standardised predictors, "one unit" is one standard deviation, which makes the coefficients comparable.

We classify ROCK1 compounds as active at **pActivity ≥ 6.5** (IC50 ≤ about 316 nM), the threshold the course uses throughout, and model the probability of activity from the eleven descriptors.



In [ ]:
y_cls <- as.integer(exact$pActivity >= 6.5)
Z <- scale(as.matrix(exact[, ..DESC]))
logit <- glm(y_cls ~ Z, family = binomial)
sl <- summary(logit)
ors <- data.table(descriptor = DESC, coef = coef(logit)[-1],
                  odds_ratio = exp(coef(logit)[-1]),
                  or_low = exp(coef(logit)[-1] - 1.96 * sl$coefficients[-1, 2]),
                  or_high = exp(coef(logit)[-1] + 1.96 * sl$coefficients[-1, 2]),
                  p = sl$coefficients[-1, 4])[order(odds_ratio)]
p <- ggplot(ors, aes(odds_ratio, factor(descriptor, levels = descriptor))) +
  geom_errorbarh(aes(xmin = or_low, xmax = or_high), height = 0.2, colour = PAL["blue"]) +
  geom_point(colour = PAL["blue"], size = 1.8) +
  geom_vline(xintercept = 1, colour = PAL["orange"], linetype = 2) +
  scale_x_log10() +
  labs(x = "odds ratio per 1 SD increase (95 % CI, log scale)", y = NULL,
       title = "Which descriptors raise the odds of pActivity >= 6.5?")
print(p)
cat(sprintf("n = %d | actives %d (%.1f %%) | McFadden pseudo-R2 = %.3f | LR test p = %.3g\n",
            length(y_cls), sum(y_cls), 100 * mean(y_cls), 1 - logit$deviance / logit$null.deviance,
            pchisq(logit$null.deviance - logit$deviance, length(DESC), lower.tail = FALSE)))
cat(sprintf("events per variable (EPV) = %.0f  (classical minimum 10)\n",
            min(sum(y_cls), sum(1 - y_cls)) / length(DESC)))
knitr::kable(ors, digits = 3)



<p class="caption"><b>Figure 7.</b> Odds ratios per one-standard-deviation increase in each descriptor, with 95 % intervals, on a logarithmic axis. The number of rotatable bonds has the largest effect — 2.26 times the odds of being active per standard deviation (95 % CI 1.90 to 2.70) — followed by QED (1.58) and the hydrogen-bond donor count (1.36); the acceptor count (0.73) and lipophilicity (0.82) push the other way. Intervals that cross the dashed line at 1 belong to descriptors with no detectable association, which here includes molecular weight, heavy atoms, TPSA, aromatic rings and FractionCSP3. The McFadden pseudo-R² of 0.087 says what the forest plot cannot: taken together these descriptors explain very little, and the model is nowhere near a usable classifier.</p>

The **events-per-variable** figure printed above the table is the classical sample-size rule for logistic regression: at least ten events (members of the rarer class) per estimated coefficient, relaxed in later work but still a useful floor<sup class="c"><a href="#r14">[14]</a></sup><sup class="c"><a href="#r15">[15]</a></sup>. Here we have 122, which is why the intervals are narrow.

### 8.1 Separation

If some combination of predictors perfectly predicts the outcome, the likelihood has no maximum: the coefficient runs to infinity and the algorithm either fails or returns an enormous estimate with an enormous standard error<sup class="c"><a href="#r16">[16]</a></sup>. This happens routinely in small datasets with a strong predictor, and the output is easy to misread as a wonderful result.



In [ ]:
set.seed(0)
sub60 <- exact[sample(.N, 60)]
y60 <- as.integer(sub60$pActivity >= 6.5)
set.seed(0)
x_sep <- y60 * 5 + rnorm(60, 0, 0.01)              ## a predictor that separates perfectly
bad <- withCallingHandlers(glm(y60 ~ x_sep, family = binomial),
                           warning = function(w) { cat("glm warning:", conditionMessage(w), "\n")
                                                   invokeRestart("muffleWarning") })
cat(sprintf("unregularised fit: coefficient %.1f, standard error %.1f, p = %.3f\n",
            coef(bad)[2], summary(bad)$coefficients[2, 2], summary(bad)$coefficients[2, 4]))
pen <- glmnet(cbind(x_sep, 0), y60, family = "binomial", alpha = 0, lambda = 0.05, standardize = FALSE)
cat(sprintf("L2-penalised fit (glmnet, lambda = 0.05): coefficient %.3f  (finite, shrunken)\n",
            as.numeric(coef(pen))[2]))
cat(sprintf("separation check: are the two classes disjoint on this predictor? %s\n",
            min(x_sep[y60 == 1]) > max(x_sep[y60 == 0])))
real <- glm(y60 ~ sub60$MolLogP, family = binomial)
cat(sprintf("for comparison, a real descriptor (MolLogP) on the same 60 compounds:\n  coefficient %.3f, SE %.3f, p = %.3f\n",
            coef(real)[2], summary(real)$coefficients[2, 2], summary(real)$coefficients[2, 4]))



R's `glm` does not stop; it warns that "fitted probabilities numerically 0 or 1 occurred", hits its iteration limit and reports a coefficient of 10.6 with a standard error of 18,560 and p = 1.000. That combination — a large coefficient with an astronomic standard error and a p-value of one — *is* the diagnostic, and it is the behaviour to recognise, because a reader skimming only the coefficient column would see a huge effect. The L2-penalised fit returns a finite, shrunken coefficient of 1.41. Both Firth's bias-reduced likelihood<sup class="c"><a href="#r17">[17]</a></sup> (the `brglm2` package, not installed here) and a simple ridge penalty are accepted remedies. A genuine descriptor on the same 60 compounds gives a small, non-significant coefficient (−0.087, p = 0.70) for comparison. If a coefficient in your logistic model exceeds about 10 in magnitude, check for separation before interpreting it.

### 8.2 Calibration

A classifier can rank perfectly and still produce meaningless probabilities. **Calibration** asks whether, among the compounds the model gives a 70 % probability of being active, about 70 % really are. It is assessed by binning the predicted probabilities and plotting the observed frequency against the mean prediction in each bin<sup class="c"><a href="#r18">[18]</a></sup>. R has no single calibration function, so the ten-line version below is the whole method — binning, averaging, plotting.



In [ ]:
p_hat <- predict(logit, type = "response")
p_bad <- pmin(pmax(p_hat * 1.6 - 0.15, 0.001), 0.999)      ## same ranking, distorted probabilities
brier <- function(p, y) mean((p - y)^2)
calib <- function(p, y, nbins = 10) {
  b <- cut(p, quantile(p, seq(0, 1, length.out = nbins + 1)), include.lowest = TRUE)
  data.table(mean_pred = as.numeric(tapply(p, b, mean)), frac = as.numeric(tapply(y, b, mean)))
}
cc <- rbind(cbind(calib(p_hat, y_cls), model = sprintf("logistic model (Brier %.3f)", brier(p_hat, y_cls))),
            cbind(calib(p_bad, y_cls), model = sprintf("same ranking, distorted (Brier %.3f)", brier(p_bad, y_cls))))
p <- ggplot(cc, aes(mean_pred, frac, colour = model)) +
  geom_abline(linetype = 2) + geom_line() + geom_point(size = 1.5) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "mean predicted probability in the bin", y = "observed fraction of actives",
       title = "Calibration: do the probabilities mean what they say?")
print(p)
cat(sprintf("logistic model : AUC %.3f, Brier %.3f\n",
            as.numeric(auc(roc(y_cls, p_hat, quiet = TRUE))), brier(p_hat, y_cls)))
cat(sprintf("distorted      : AUC %.3f, Brier %.3f  <- identical ranking, worse probabilities\n",
            as.numeric(auc(roc(y_cls, p_bad, quiet = TRUE))), brier(p_bad, y_cls)))
cat(sprintf("base rate of actives: %.3f | mean predicted probability: %.3f\n", mean(y_cls), mean(p_hat)))



<p class="caption"><b>Figure 8.</b> Calibration curves. The logistic model (blue) tracks the diagonal closely — unsurprisingly, since maximum-likelihood logistic regression fitted and evaluated on the same data is calibrated by construction, and its mean predicted probability equals the base rate of 0.484 exactly. The orange curve is the <em>same model's ranking</em> with the probabilities stretched; its ROC-AUC is identical to three decimals (0.705) because AUC depends only on the ordering, while its Brier score is clearly worse (0.243 against 0.220) and its curve departs from the diagonal at both ends. A model can therefore be a perfect ranker and a useless probability estimate, which matters whenever a predicted probability feeds a decision — a screening budget, a triage threshold, a cost calculation.</p>

Two cautions. Calibration measured on the training data is optimistic; the honest version uses held-out predictions, which Unit 12 builds. And a calibration curve says nothing about discrimination: a model that predicts the base rate for every compound is perfectly calibrated and completely useless.

## 9. Regularisation: ridge, lasso and elastic net

So far we used eleven hand-picked descriptors. Real descriptor sets are much larger, and with p approaching n ordinary least squares breaks down: the variance of \( \hat\beta \) grows without bound as \( X^{\mathsf T}X \) approaches singularity. Regularisation adds a penalty on the size of the coefficients, accepting a little bias in exchange for a large reduction in variance:

\[ \text{ridge:}\ \min_\beta \lVert y-X\beta\rVert^2 + \lambda\lVert\beta\rVert_2^2,\qquad \text{lasso:}\ \min_\beta \lVert y-X\beta\rVert^2 + \lambda\lVert\beta\rVert_1, \]

with the **elastic net** mixing the two. Ridge<sup class="c"><a href="#r19">[19]</a></sup> shrinks all coefficients smoothly towards zero and keeps every predictor; the lasso<sup class="c"><a href="#r20">[20]</a></sup> sets some exactly to zero, performing selection; the elastic net<sup class="c"><a href="#r21">[21]</a></sup> keeps groups of correlated predictors together instead of picking one arbitrarily. The geometry is the explanation: the L1 constraint region has corners on the axes, and a corner is where the solution tends to land. In R all three are `glmnet`<sup class="c"><a href="#r35">[35]</a></sup> with `alpha` = 0, 1 or something in between.

<div class="box warn"><h4>Substitution in the R edition — the high-dimensional descriptor set</h4>
<p>The Python edition builds this section on about 200 RDKit whole-molecule descriptors computed in the notebook. R has no RDKit binding here, so the high-dimensional design matrix is built instead from the <strong>frozen Morgan fingerprints</strong> (radius 2, 2,048 bits) in <code>data/rock1_morgan_sparse.csv.gz</code>, documented in <a href="../../data/manifest_fingerprints.json">data/manifest_fingerprints.json</a>, keeping the bits that are set in at least 10 compounds, plus the 11 standardised descriptors. That gives 1,325 predictors for 2,764 compounds — the same p-approaching-n regime the Python unit demonstrates, with substructure indicators instead of computed descriptors. Expect the <em>predictive</em> numbers to be better than the Python edition's for that reason: fingerprint bits say which substructures a molecule contains, which is far more informative about potency than its molecular weight.</p></div>



In [ ]:
fp <- fread(D("rock1_morgan_sparse.csv.gz"))[id %in% exact$inchikey]
M  <- sparseMatrix(i = match(fp$id, exact$inchikey), j = fp$bit + 1L, x = 1,
                   dims = c(nrow(exact), 2048))
kb <- which(colSums(M) >= 10)                         ## bits set in at least 10 compounds
Xbig <- cbind(scale(as.matrix(exact[, ..DESC])), as.matrix(M[, kb]))
colnames(Xbig) <- c(DESC, paste0("bit_", kb - 1))
ybig <- exact$pActivity
cat(sprintf("fingerprint bits set in >= 10 compounds: %d (mean %.1f bits per molecule)\n",
            length(kb), mean(rowSums(M))))
cat(sprintf("design matrix: %d compounds x %d predictors (p/n = %.2f)\n",
            nrow(Xbig), ncol(Xbig), ncol(Xbig) / nrow(Xbig)))


In [ ]:
set.seed(0)
folds <- sample(rep(1:5, length.out = nrow(Xbig)))
cv_r2 <- function(predfun) {
  sapply(1:5, function(k) {
    tr <- folds != k; te <- folds == k
    pr <- predfun(Xbig[tr, ], ybig[tr], Xbig[te, ])
    1 - sum((ybig[te] - pr)^2) / sum((ybig[te] - mean(ybig[tr]))^2)
  })
}
ols_pred <- function(xtr, ytr, xte) {
  f <- lm.fit(cbind(1, xtr), ytr); b <- f$coefficients; b[is.na(b)] <- 0
  as.vector(cbind(1, xte) %*% b)
}
set.seed(0); cv_ridge <- cv.glmnet(Xbig, ybig, alpha = 0,   nfolds = 5)
set.seed(0); cv_lasso <- cv.glmnet(Xbig, ybig, alpha = 1,   nfolds = 5)
set.seed(0); cv_enet  <- cv.glmnet(Xbig, ybig, alpha = 0.5, nfolds = 5)
rows <- list()
f_ols <- lm.fit(cbind(1, Xbig), ybig)
r <- cv_r2(ols_pred)
rows[[1]] <- data.table(model = "OLS (no penalty)",
                        train_R2 = 1 - sum(f_ols$residuals^2) / sum((ybig - mean(ybig))^2),
                        CV_R2 = mean(r), CV_sd = sd(r), nonzero = sum(!is.na(f_ols$coefficients)) - 1)
for (z in list(list("Ridge", 0, cv_ridge), list("Lasso", 1, cv_lasso), list("Elastic net", 0.5, cv_enet))) {
  a <- z[[2]]; lam <- z[[3]]$lambda.min
  r <- cv_r2(function(xtr, ytr, xte) as.vector(predict(glmnet(xtr, ytr, alpha = a, lambda = lam), xte)))
  g <- glmnet(Xbig, ybig, alpha = a, lambda = lam)
  rows[[length(rows) + 1]] <- data.table(
    model = z[[1]],
    train_R2 = 1 - sum((ybig - as.vector(predict(g, Xbig)))^2) / sum((ybig - mean(ybig))^2),
    CV_R2 = mean(r), CV_sd = sd(r), nonzero = sum(coef(g)[-1] != 0))
}
cat(sprintf("chosen penalties: ridge lambda %.4f | lasso %.4f | elastic net %.4f (alpha 0.5)\n",
            cv_ridge$lambda.min, cv_lasso$lambda.min, cv_enet$lambda.min))
knitr::kable(rbindlist(rows), digits = 3)



The table is the bias–variance trade-off in four rows. Ordinary least squares fits the training data best — it always does, it is the unbiased minimiser of training error, reaching R² = 0.821 — and its **cross-validated R² is −0.506**, meaning it predicts held-out compounds considerably worse than the mean of the training set does. With 1,325 correlated predictors and 2,764 compounds, the unpenalised solution is fitting noise. The penalised models recover completely: ridge, lasso and elastic net all reach a cross-validated R² of about 0.50 with a fold-to-fold standard deviation of 0.017, ten times smaller than OLS's 0.166, and they do it with training R² around 0.66 — *worse* on the training data and vastly better on new compounds, which is what accepting bias buys. The lasso reaches that with 430 of the 1,325 predictors non-zero and the elastic net with 438, while ridge keeps all of them.

Negative cross-validated R² is the single clearest symptom of over-fitting, and it is invisible to anyone who reports only the training R². This is also the first appearance in the course of the rule that governs all of Block D: **the penalty must be chosen inside the cross-validation, not on the test data**.



In [ ]:
path <- glmnet(Xbig, ybig, alpha = 1)
op <- par(mfrow = c(1, 2), mar = c(4.2, 4, 2.6, 1), cex = 0.72)
plot(path, xvar = "lambda", label = FALSE, col = "#44444455",
     main = "lasso coefficient paths (1,325 predictors)")
abline(v = log(cv_lasso$lambda.min), col = PAL["orange"], lty = 2, lwd = 1.6)
plot(cv_lasso, main = "the CV curve that picked it")
par(op)
for (l in c(0.2, 0.1, cv_lasso$lambda.min, 0.005)) {
  k <- which.min(abs(path$lambda - l))
  cat(sprintf("lambda = %.4f -> %4d non-zero coefficients\n", path$lambda[k], path$df[k]))
}
cat(sprintf("CV-optimal lambda = %.4f (CV MSE %.4f); lambda.1se = %.4f with %d non-zero\n",
            cv_lasso$lambda.min, min(cv_lasso$cvm), cv_lasso$lambda.1se,
            sum(coef(glmnet(Xbig, ybig, alpha = 1, lambda = cv_lasso$lambda.1se))[-1] != 0)))



<p class="caption"><b>Figure 9.</b> Left: each line is one predictor's coefficient as the lasso penalty is relaxed from strong (right) to weak (left); at log λ ≈ −1.2 everything is zero and predictors enter one by one as λ decreases — 7 at λ = 0.204, 39 at λ = 0.097, 431 at the cross-validated optimum of 0.0114. Right: `glmnet`'s own cross-validation plot, with the mean squared error and its fold-to-fold standard error against log λ, the number of non-zero coefficients across the top, and two vertical lines: λ<sub>min</sub> (CV MSE 0.615) and λ<sub>1se</sub> = 0.0166, the largest penalty within one standard error of the best, which keeps 328 predictors instead of 430. That "one standard error rule" is built into R's output precisely because the curve is flat near its minimum.</p>

### 9.1 Is the lasso's selection stable?

A lasso coefficient path tells you *which* predictors were selected on *this* sample. It is tempting to read the selected set as the chemistry. The test of that reading is to resample the compounds and see whether the same predictors keep being selected — the idea behind stability selection<sup class="c"><a href="#r22">[22]</a></sup>.



In [ ]:
lam_fix <- cv_lasso$lambda.min
B <- 60
set.seed(0)
sel_count <- numeric(ncol(Xbig))
for (b in 1:B) {
  ii <- sample(nrow(Xbig), replace = TRUE)
  g  <- glmnet(Xbig[ii, ], ybig[ii], alpha = 1, lambda = lam_fix)
  sel_count <- sel_count + (as.vector(coef(g)[-1]) != 0)
}
full <- glmnet(Xbig, ybig, alpha = 1, lambda = lam_fix)
freq <- data.table(predictor = colnames(Xbig), frequency = sel_count / B,
                   coef_full = as.vector(coef(full)[-1]))[order(-frequency)]
p <- ggplot(freq, aes(frequency)) +
  geom_histogram(bins = 20, fill = "grey75", colour = NA) +
  geom_vline(xintercept = 0.8, colour = PAL["orange"], linetype = 2) +
  labs(x = sprintf("fraction of %d bootstrap resamples in which the predictor was selected", B),
       y = "predictors", title = "Most selected predictors are not reproducibly selected")
print(p)
cat(sprintf("penalty fixed at lambda = %.4f -> %d non-zero coefficients on the full data\n",
            lam_fix, sum(coef(full)[-1] != 0)))
cat(sprintf("selected in >= 80 %% of resamples : %d\n", sum(freq$frequency >= 0.8)))
cat(sprintf("selected in <= 20 %% of resamples : %d\n", sum(freq$frequency <= 0.2)))
cat(sprintf("in between (the dangerous band)  : %d\n",
            sum(freq$frequency > 0.2 & freq$frequency < 0.8)))
cat("\nselection frequency of the 11 interpretable descriptors:\n")
knitr::kable(freq[predictor %in% DESC], digits = 3)



<p class="caption"><b>Figure 10.</b> Distribution of selection frequency over 60 bootstrap resamples at a fixed penalty. The histogram is strongly bimodal: 130 predictors are selected in at least 80 % of resamples, 590 in a fifth of them or fewer, and 605 sit in between. The predictors in that middle band are the dangerous ones — many appear among the 430 non-zero coefficients of the model fitted to the full data and would be reported as "the features the lasso selected", yet a different sample of the same compounds would have chosen different ones. Among the eleven interpretable descriptors, only HBD (0.92) and FractionCSP3 (0.87) clear the 80 % bar, while MolWt, MolLogP, TPSA and QED are selected in no resample at all — the fingerprint bits carry their information better.</p>

The rule that follows: **report selection frequency, not a selected set**. A predictor chosen in 58 of 60 resamples is a finding; one chosen in 25 of 60 is a coin flip. When correlated predictors compete, the lasso picks one of them essentially at random, which is exactly the situation the elastic net was designed for<sup class="c"><a href="#r21">[21]</a></sup>.

<div class="exercise"><h4>Exercise 2</h4>
<p>Repeat the stability experiment with <code>alpha = 0.5</code> (elastic net) instead of <code>alpha = 1</code> at the same λ. Does the number of predictors selected in at least 80 % of resamples go up or down, and why? Then take the ten most stable fingerprint bits, find the compounds in which each is set (<code>which(M[, bit + 1] == 1)</code>), and check whether the bits that survive are the ones shared by the large scaffold series — i.e. whether the "stable" features are chemistry or series membership.</p></div>

## 10. Comparing models honestly

Five tools, each answering a different question:

| tool | compares | penalises complexity? | valid for |
|---|---|---|---|
| R² | fit to the training data | no | nothing — always increases with more predictors |
| adjusted R² | fit, with a degrees-of-freedom penalty | mildly | nested or non-nested linear models |
| AIC<sup class="c"><a href="#r23">[23]</a></sup> | expected predictive loss | yes, 2 per parameter | models on the **same data**, nested or not |
| BIC<sup class="c"><a href="#r24">[24]</a></sup> | approximate posterior probability | yes, log(n) per parameter | same, but much stricter for large n |
| nested F / likelihood-ratio test | two **nested** models | via the degrees of freedom | nested models only |



In [ ]:
specs <- c("intercept only"     = "pActivity ~ 1",
           "+ MolLogP"          = "pActivity ~ MolLogP",
           "+ MolLogP + TPSA"   = "pActivity ~ MolLogP + TPSA",
           "+ 5 descriptors"    = "pActivity ~ MolLogP + TPSA + HBD + HBA + RotB",
           "all 9 non-constant" = paste("pActivity ~", paste(usable, collapse = " + ")))
fits <- lapply(specs, function(f) lm(as.formula(f), data = series))
cmp <- rbindlist(lapply(seq_along(fits), function(i) {
  f <- fits[[i]]
  dt <- data.table(model = names(specs)[i], k = length(coef(f)), R2 = summary(f)$r.squared,
                   adj_R2 = summary(f)$adj.r.squared, AIC = AIC(f), BIC = BIC(f))
  if (i > 1) { an <- anova(fits[[i - 1]], f); dt[, `:=`(F_vs_prev = an[2, 5], p_vs_prev = an[2, 6])] }
  dt
}), fill = TRUE)
p <- ggplot(melt(cmp[, .(k, AIC, BIC)], id.vars = "k"), aes(k, value, colour = variable)) +
  geom_line() + geom_point(size = 1.8) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "number of estimated parameters", y = "information criterion",
       title = "AIC and BIC price complexity differently")
print(p)
cat(sprintf("best by adjusted R2: %s\n", cmp$model[which.max(cmp$adj_R2)]))
cat(sprintf("best by AIC        : %s\n", cmp$model[which.min(cmp$AIC)]))
cat(sprintf("best by BIC        : %s\n", cmp$model[which.min(cmp$BIC)]))
knitr::kable(cmp, digits = 4)



<p class="caption"><b>Figure 11.</b> AIC and BIC for five nested models on the same 207 compounds, against the number of estimated parameters. The two criteria disagree completely: AIC is minimised by the two-descriptor model (502.9) and rises to 505.4 for the nine-descriptor one, while BIC is minimised by the intercept-only model (511.0) and rises monotonically to 542.1. Adjusted R² prefers the largest model, whose nested F test against the five-descriptor model is the only one in the table to reach p &lt; 0.05 (0.042) — a reminder that with nine predictors on a weak signal, "best" depends entirely on how complexity is priced.</p>

The three criteria need not agree, and when they disagree the reason is always the same: they price complexity differently. BIC's penalty of log(n) per parameter — here log(207) = 5.3 — is harsher than AIC's 2 per parameter for any n above 7, so BIC prefers smaller models, and the gap widens with sample size. Adjusted R² is the mildest penalty of the three and will often keep a predictor that AIC drops.

Three rules for using them. **AIC and BIC are comparable only between models fitted to exactly the same observations** — drop a row because of a missing descriptor value and the comparison is void, which in R is easy to do by accident because `lm` silently applies `na.omit`. **Absolute values are meaningless**; only differences matter, and a difference below about 2 is not evidence. And **the nested F test (or its likelihood-ratio equivalent) applies only to nested models**: it is the formal version of "does adding these parameters improve the fit more than chance would".

## 11. Confounding, adjustment and blocking

A **confounder** is a variable associated with both the predictor of interest and the response, which can create or hide an association. The standard remedy in a linear model is to include it as a covariate; when the confounder is the identity of the subject, including it is called **blocking**<sup class="c"><a href="#r9">[9]</a></sup>.

<div class="box bridge"><h4>Domain context — what the periodontitis table does and does not contain</h4>
<p>The gingival-tissue expression dataset<sup class="c"><a href="#r25">[25]</a></sup> contains, for each of 69 patients, one sample from a site affected by periodontitis — a chronic inflammatory disease of the tissues around the teeth — and one from a clinically healthy site, with log<sub>2</sub> expression values for 5,000 genes. <strong>This series carries no demographic covariates</strong> — the GEO record provides only tissue and phenotype, so there is no age, sex, smoking status or tooth type to adjust for. The confounding example below therefore uses <code>patient_id</code> as a blocking factor, which is the strongest available adjustment and, in a within-patient design, the most relevant one: it absorbs every characteristic of the patient, measured or not, including the demographic variables we do not have. When a dataset does carry demographics — as most clinical tables the group works with do — the same <code>+ age + sex + smoking</code> syntax adds them, and the interpretation of the coefficient of interest becomes "adjusted for" those variables.</p></div>



In [ ]:
expr <- fread(D("geo_periodontitis_expr.csv"))
meta <- fread(D("geo_periodontitis_meta.csv"))
dis <- meta[status == "diseased"][order(patient_id)]
hea <- meta[status == "healthy"][order(patient_id)]
GENE <- "PLA2R1"
rowg <- expr[symbol == GENE]
long <- data.table(
  y = c(as.numeric(rowg[, dis$sample_id, with = FALSE]), as.numeric(rowg[, hea$sample_id, with = FALSE])),
  diseased = c(rep(1, nrow(dis)), rep(0, nrow(hea))),
  patient = factor(c(dis$patient_id, hea$patient_id)))
m_plain <- lm(y ~ diseased, data = long)
m_block <- lm(y ~ diseased + patient, data = long)
bars <- rbindlist(lapply(list(list("no adjustment", m_plain), list("blocked on patient", m_block)), function(z) {
  m <- z[[2]]; ci <- confint(m)["diseased", ]
  data.table(model = z[[1]], beta = coef(m)["diseased"], lo = ci[1], hi = ci[2],
             se = summary(m)$coefficients["diseased", 2], p = summary(m)$coefficients["diseased", 4])
}))
p <- ggplot(bars, aes(beta, model)) +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.15, colour = PAL["blue"]) +
  geom_point(colour = PAL["blue"], size = 2) +
  geom_vline(xintercept = 0, linetype = 2) +
  geom_text(aes(x = hi, label = sprintf("SE %.3f, p = %.1e", se, p)), hjust = -0.08, size = 2.6) +
  coord_cartesian(xlim = c(-1.3, -0.4)) +
  labs(x = sprintf("estimated effect of disease on log2 expression of %s (95 %% CI)", GENE), y = NULL,
       title = "Blocking does not move the estimate; it sharpens it")
print(p)
cat(sprintf("unadjusted : beta = %+.3f, SE = %.3f, p = %.2e, R2 = %.3f\n",
            coef(m_plain)["diseased"], summary(m_plain)$coefficients["diseased", 2],
            summary(m_plain)$coefficients["diseased", 4], summary(m_plain)$r.squared))
cat(sprintf("blocked    : beta = %+.3f, SE = %.3f, p = %.2e, R2 = %.3f (%d extra parameters)\n",
            coef(m_block)["diseased"], summary(m_block)$coefficients["diseased", 2],
            summary(m_block)$coefficients["diseased", 4], summary(m_block)$r.squared,
            length(coef(m_block)) - length(coef(m_plain))))
cat(sprintf("standard error reduced by %.1f %%\n",
            100 * (1 - summary(m_block)$coefficients["diseased", 2] /
                     summary(m_plain)$coefficients["diseased", 2])))
cat(sprintf("AIC: %.1f (unadjusted) vs %.1f (blocked) | BIC: %.1f vs %.1f\n",
            AIC(m_plain), AIC(m_block), BIC(m_plain), BIC(m_block)))



<p class="caption"><b>Figure 12.</b> The estimated effect of disease on the expression of one gene, with and without the patient blocking factor. The point estimate is identical to three decimals, −0.980 log<sub>2</sub> units — because the design is balanced, every patient contributing exactly one sample of each type — while the standard error falls from 0.102 to 0.074, a reduction of 27.5 %, and the interval narrows from (−1.181, −0.779) to (−1.127, −0.833). Blocking in a balanced design buys precision, not a different answer; in an <em>unbalanced</em> design it would also move the estimate, and that movement is what "adjusting for a confounder" means.</p>

The AIC and BIC printed below the figure disagree sharply about whether the blocking factor is worth its 68 extra parameters: AIC prefers the blocked model (204.7 against 253.1), BIC prefers the simple one (261.9 against 412.5). Neither is wrong — they answer different questions, and the design, not an information criterion, decides here. The samples are paired by construction, so the blocked model is the correct one regardless of what a criterion says.



In [ ]:
suppressPackageStartupMessages(library(lme4))
mm <- lmer(y ~ diseased + (1 | patient), data = long, REML = TRUE)
vc <- as.data.frame(VarCorr(mm))
cat(sprintf("lmer fixed effect of disease: %+.3f (SE %.3f)  <- compare lm with 68 dummies: %+.3f (SE %.3f)\n",
            fixef(mm)["diseased"], sqrt(diag(vcov(mm)))[2],
            coef(m_block)["diseased"], summary(m_block)$coefficients["diseased", 2]))
cat(sprintf("variance components: between-patient SD %.3f | residual SD %.3f -> ICC %.2f\n",
            vc$sdcor[1], vc$sdcor[2], vc$vcov[1] / sum(vc$vcov)))
cat(sprintf("parameters: mixed model %d fixed + 1 variance | fixed-effects model %d\n",
            length(fixef(mm)), length(coef(m_block))))



<div class="box warn"><h4>Fixed effects or mixed models?</h4>
<p>Adding 68 patient dummies spends 68 degrees of freedom to estimate nuisance parameters nobody will read. The economical alternative is a <strong>mixed-effects model</strong>, which treats the patient effect as a random draw from a distribution and estimates its variance rather than its 69 values. The chunk above fits <code>lmer(y ~ diseased + (1 | patient))</code><sup class="c"><a href="#r36">[36]</a></sup> and gets the same coefficient, −0.980, with the same standard error, 0.074, from 2 fixed parameters and one variance component instead of 70 coefficients; the estimated between-patient SD of 0.411 against a residual SD of 0.433 gives an intraclass correlation of 0.47, i.e. nearly half the variance in this gene's expression is between patients. For a balanced paired design the two approaches agree; for unbalanced designs with many groups and few observations each — exactly the structure of compounds nested in assays, or frames nested in replicas — the mixed model is substantially better, and it is the natural frequentist sibling of the hierarchical Bayesian model in <a href="../U10_Bayesian/index.html">Unit 10</a>.</p></div>

## 12. Explaining or predicting?

Two cultures use the same equation for different purposes<sup class="c"><a href="#r26">[26]</a></sup><sup class="c"><a href="#r27">[27]</a></sup>:

- **Explanatory modelling** asks what the coefficients are. It needs correct specification, interpretable predictors, attention to collinearity and confounding, and intervals on the coefficients. Its failure mode is a confident causal story from an observational fit — a regression coefficient is an association, and no amount of adjustment turns an observational association into a cause<sup class="c"><a href="#r31">[31]</a></sup>.
- **Predictive modelling** asks how accurate predictions are on new data. It tolerates collinearity, black-box features and uninterpretable parameters, and is judged only by held-out performance. Its failure mode is a model that predicts well and is read as if it explained something.

The same dataset answers the two questions differently, which this unit has demonstrated twice. The eleven-descriptor model fitted within a congeneric series (§3) is an explanatory attempt, and it says there is nothing to explain with whole-molecule descriptors: R² = 0.078, F p = 0.059. The 1,325-predictor penalised model (§9) is a predictive attempt, and it reaches a cross-validated R² of 0.50 — genuinely useful for ranking new compounds — while its *selected features are not a chemical conclusion* (§9.1). Writing the first result in a paper as if the second had supported it is the single most common misuse of QSAR.

<div class="box key"><h4>Decide before you fit</h4>
<p>Write one sentence stating whether the model exists to produce a number for new compounds or to support a statement about which properties matter. The decision determines the descriptor set, the validation protocol, the diagnostics that matter, and what may be written in the abstract. If the answer is "both", build two models and report them separately.</p></div>

<div class="box bridge"><h4>Bridge</h4>
<p>This unit completes the statistics block that began with <a href="../U7_Estimation/index.html">Unit 7</a> (intervals, bootstrap, noise floor) and <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a> (tests, effect sizes, multiplicity, power); <a href="../U10_Bayesian/index.html">Unit 10</a> re-fits the hierarchical version of §11 with explicit priors and compares it with the <code>lmer</code> fit above. The Python counterpart of this unit is <a href="../../Python/U9_Linear_Models/index.html">here</a> and the R edition is indexed at <a href="../index.html">the course root</a>. Forward, <a href="../U11_Supervised_Workflow/index.html">Unit 11</a> turns the regularised fit of §9 into a leakage-free pipeline, <a href="../U12_Validation/index.html">Unit 12</a> supplies the split taxonomy and y-randomisation that §7 previewed, <a href="../U13_Features_Dimensionality/index.html">Unit 13</a> develops descriptor selection beyond the lasso, <a href="../U16_Interpretability/index.html">Unit 16</a> asks what a non-linear model learned, and <a href="../U21_Omics/index.html">Unit 21</a> fits one linear model per gene with moderated variances<sup class="c"><a href="#r28">[28]</a></sup>. In the sister courses, the cheminformatics unit on <a href="../../../Cheminformatics/U12_Classical_QSAR/index.html">classical QSAR</a> covers the chemistry of Hansch analysis and its substituent constants, the unit on <a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">descriptors and fingerprints</a> explains what the fingerprint bits of §9 encode, and the R edition of the AI course, indexed <a href="../../../BGM411/R/index.html">here</a>, derives the same estimators from the machine-learning side.</p></div>

## 13. Summary

You should be able to explain, without notes:

- the linear model's four assumptions in order of importance, and which diagnostic detects each;
- how to compute OLS from the normal equations and where the standard error of a coefficient comes from;
- how to read a regression table in the right order: F test and R² first, coefficients last, and why `NA` coefficients are R telling you about aliased columns;
- what the four panels of `plot(fit)` each detect, and the difference between leverage and influence;
- what a variance inflation factor measures (QED reached 24.9 inside the series), why a constant descriptor must be removed, and the three responses to collinearity;
- that ANOVA is a linear model with categorical predictors — the same F = 31.87 from `anova()` and `oneway.test()` — and what an interaction term estimates;
- why R² = 0.595 on 33 compounds with 9 descriptors is mostly chance (the scrambled mean was 0.281), and how y-randomisation quantifies it;
- how to interpret a logistic coefficient as an odds ratio (RotB: 2.26 per SD), how to recognise separation (coefficient 10.6 with SE 18,560), and why a well-ranking model can be badly calibrated;
- how ridge, lasso and elastic net trade bias for variance, why unpenalised OLS had a cross-validated R² of −0.506 where the penalised fits reached +0.50, and why lasso selection must be checked for stability;
- when to use adjusted R², AIC, BIC and the nested F test, and that blocking reduced the standard error by 27.5 % without moving the estimate;
- the difference between explanatory and predictive modelling, and why the choice must be made before fitting.

## 14. Self-assessment

<details><summary>1. Your R² rises every time you add a descriptor. Is the model improving?</summary>No. R² cannot decrease when a predictor is added, even a column of random numbers, because the least-squares fit can always set the new coefficient to zero and do no worse — in §3, sixty noise columns reached R² = 0.267 on 207 compounds. Use adjusted R² (which ended at −0.034 there), AIC or cross-validated R², all of which can go down.</details>

<details><summary>2. A coefficient has a huge standard error and a VIF of 40. What is happening and what do you do?</summary>That descriptor is nearly a linear combination of the others, so the data cannot separate their contributions. Drop or merge the redundant descriptors on chemical grounds, or regularise. Do not conclude that the descriptor is unimportant — collinearity inflates the standard error without changing the predictions.</details>

<details><summary>3. What does leverage measure, and how does it differ from influence?</summary>Leverage (<code>hatvalues</code>) measures how far a compound's predictor values are from the centre of the descriptor cloud; it depends on X only. Influence (<code>cooks.distance</code>) measures how much the fitted model changes if that compound is removed, combining leverage with the size of its residual. A high-leverage compound that fits well has little influence.</details>

<details><summary>4. Your QSAR on 25 compounds with 8 descriptors gives R² = 0.75. What must you report alongside it?</summary>The adjusted R² (which falls sharply with 8 parameters on 25 points), a y-randomisation control with the distribution of scrambled R², and an external or cross-validated estimate. With about 3 compounds per descriptor the expected R² under pure noise is already p/(n−1) = 0.33.</details>

<details><summary>5. Why is the ring count constant inside a congeneric series, and what does R do about it?</summary>Because a congeneric series is defined by a shared scaffold, and the rings are part of the scaffold. A constant column carries no information and is collinear with the intercept; R's QR pivoting detects the aliasing, drops the column and prints <code>NA</code> for its coefficient — which is how we discovered that two of the eleven descriptors were constant. <code>car::vif()</code> then refuses to run until they are removed by hand.</details>

<details><summary>6. Interpret a logistic coefficient of 0.816 on a standardised descriptor.</summary>The odds of being active multiply by exp(0.816) = 2.26 for each one-standard-deviation increase in that descriptor (here the rotatable-bond count), holding the others fixed. Report the odds ratio with its confidence interval (1.90 to 2.70), not the coefficient, and state that "holding the others fixed" is an arithmetic statement about the model, not a claim that the descriptor can be changed independently in a real molecule.</details>

<details><summary>7. Your logistic model reports a coefficient of 10.6 with a standard error of 18,560 and p = 1.000. What happened?</summary>Separation: some predictor or combination of predictors perfectly classifies the outcome, so the likelihood has no finite maximum and <code>glm</code> stops at its iteration limit with the warning "fitted probabilities numerically 0 or 1 occurred". Check whether the classes are disjoint on that predictor, and use a penalised (<code>glmnet</code>) or Firth-corrected fit.</details>

<details><summary>8. Model A has AUC 0.705 and Brier 0.220; model B has AUC 0.705 and Brier 0.243. Which do you deploy?</summary>A, if the predicted probabilities are used for anything — a cost calculation, a triage threshold, a screening budget. The two models rank compounds identically (that is what the equal AUC means); A's probabilities are closer to the observed frequencies. If only the ranking is used, the choice is indifferent.</details>

<details><summary>9. Why can unpenalised OLS have a negative cross-validated R²?</summary>Because cross-validated R² is computed on data the model has not seen, and a model with 1,325 correlated predictors fits the noise of the training set so closely (train R² 0.821) that its predictions on new compounds are worse than simply predicting the training mean (CV R² −0.506). Negative CV R² is the signature of over-fitting, and it is invisible if only the training R² is reported.</details>

<details><summary>10. The lasso selected 430 predictors. Can you write "these 430 features determine potency"?</summary>No. Repeat the fit on bootstrap resamples: in our experiment only 130 predictors were selected in at least 80 % of resamples, while 590 appeared in a fifth or fewer and 605 sat in the unstable middle. Report selection frequencies, and remember that when predictors are correlated, the lasso picks one of them nearly arbitrarily.</details>

<details><summary>11. AIC prefers the two-descriptor model and BIC the intercept-only one. Which do you believe?</summary>Neither automatically — they target different things (AIC approximates predictive loss, BIC approximates posterior model probability with a log n penalty) and disagreement simply means the extra parameters are worth about as much as they cost. Decide with the design and the scientific question; in a paired design, the blocking factor stays in the model whatever BIC says.</details>

<details><summary>12. What does blocking on patient do to the estimate and to its standard error in a balanced paired design?</summary>It leaves the point estimate unchanged (−0.980 both ways) and reduces the standard error, here from 0.102 to 0.074, because between-patient variation is removed from the residual. In an unbalanced design it would also shift the estimate, which is the sense in which it "adjusts for" the patient. A mixed model with <code>(1 | patient)</code> achieves the same with one variance parameter instead of 68 dummies.</details>

## Glossary

<dl class="gloss">
<dt>Adjusted R²</dt><dd>R² penalised for the number of predictors; unlike R² it can decrease when a predictor is added.</dd>
<dt>AIC / BIC</dt><dd>Information criteria balancing fit against complexity, with penalties of 2 and log(n) per parameter; comparable only across models on identical data.</dd>
<dt>Aliased coefficient</dt><dd>A predictor exactly collinear with others (e.g. constant within the data); R drops it and reports <code>NA</code>.</dd>
<dt>Blocking</dt><dd>Including the identity of the experimental unit (patient, batch, plate) as a factor, removing its variation from the residual.</dd>
<dt>Calibration</dt><dd>Agreement between predicted probabilities and observed frequencies; measured by a reliability curve and the Brier score.</dd>
<dt>Chance correlation</dt><dd>Apparent fit arising from too many descriptors on too few compounds; quantified by response randomisation.</dd>
<dt>Confounder</dt><dd>A variable associated with both predictor and response that can create or mask an apparent effect.</dd>
<dt>Cook's distance</dt><dd>Influence measure combining leverage and residual size; the change in the fit when one observation is deleted.</dd>
<dt>Elastic net</dt><dd>Penalty mixing L1 and L2 (<code>glmnet</code> with 0 &lt; alpha &lt; 1); keeps correlated predictors together instead of choosing one.</dd>
<dt>Hansch analysis</dt><dd>Classical QSAR: a linear model of potency in physicochemical substituent properties within a congeneric series.</dd>
<dt>Interaction</dt><dd>A product term letting the slope on one predictor depend on another; <code>x1 * x2</code> in an R formula.</dd>
<dt>Intraclass correlation</dt><dd>Share of total variance attributable to the grouping factor of a mixed model.</dd>
<dt>Lasso</dt><dd>L1-penalised regression (<code>alpha = 1</code>); shrinks some coefficients exactly to zero, performing selection.</dd>
<dt>Leverage</dt><dd>How far an observation's predictors lie from their centroid; the diagonal of the hat matrix, <code>hatvalues()</code>.</dd>
<dt>Lambda.1se</dt><dd>The largest penalty whose cross-validated error is within one standard error of the minimum; <code>cv.glmnet</code>'s conservative default.</dd>
<dt>Logistic regression</dt><dd>Linear model for the log-odds of a binary outcome; coefficients exponentiate to odds ratios.</dd>
<dt>Multicollinearity</dt><dd>Near-linear dependence among predictors; inflates coefficient variances without harming predictions.</dd>
<dt>Normal equations</dt><dd>XᵀXβ = Xᵀy, whose solution is the least-squares estimate.</dd>
<dt>Odds ratio</dt><dd>exp(coefficient) in a logistic model; the multiplicative change in odds per unit of the predictor.</dd>
<dt>Ridge regression</dt><dd>L2-penalised regression (<code>alpha = 0</code>); shrinks all coefficients smoothly and keeps every predictor.</dd>
<dt>Separation</dt><dd>A predictor combination perfectly classifying the outcome, so the logistic likelihood has no finite maximum.</dd>
<dt>Stability selection</dt><dd>Assessing which predictors a selection method keeps across resamples of the data.</dd>
<dt>VIF</dt><dd>Variance inflation factor, 1/(1 − R²ⱼ); the factor by which collinearity inflates a coefficient's variance.</dd>
<dt>y-randomisation</dt><dd>Permuting the response and refitting, to obtain the null distribution of a goodness-of-fit statistic.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r2">Hansch C, Maloney PP, Fujita T, Muir RM. Correlation of biological activity of phenoxyacetic acids with Hammett substituent constants and partition coefficients. <em>Nature</em>. 1962;194:178-180. <a href="https://doi.org/10.1038/194178b0">doi:10.1038/194178b0</a></li>
<li id="r3">Hansch C, Fujita T. p-sigma-pi analysis. A method for the correlation of biological activity and chemical structure. <em>J Am Chem Soc</em>. 1964;86(8):1616-1626. <a href="https://doi.org/10.1021/ja01062a035">doi:10.1021/ja01062a035</a></li>
<li id="r4">Topliss JG, Costello RJ. Chance correlations in structure-activity studies using multiple regression analysis. <em>J Med Chem</em>. 1972;15(10):1066-1068. <a href="https://doi.org/10.1021/jm00280a017">doi:10.1021/jm00280a017</a></li>
<li id="r5">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r6">Cook RD. Detection of influential observation in linear regression. <em>Technometrics</em>. 1977;19(1):15-18. <a href="https://doi.org/10.1080/00401706.1977.10489493">doi:10.1080/00401706.1977.10489493</a></li>
<li id="r7">O'Brien RM. A caution regarding rules of thumb for variance inflation factors. <em>Qual Quant</em>. 2007;41:673-690. <a href="https://doi.org/10.1007/s11135-006-9018-6">doi:10.1007/s11135-006-9018-6</a></li>
<li id="r8">Dormann CF, Elith J, Bacher S, et al. Collinearity: a review of methods to deal with it and a simulation study evaluating their performance. <em>Ecography</em>. 2013;36(1):27-46. <a href="https://doi.org/10.1111/j.1600-0587.2012.07348.x">doi:10.1111/j.1600-0587.2012.07348.x</a></li>
<li id="r9">Krzywinski M, Altman N. Analysis of variance and blocking. <em>Nat Methods</em>. 2014;11:699-700. <a href="https://doi.org/10.1038/nmeth.3005">doi:10.1038/nmeth.3005</a></li>
<li id="r10">Golbraikh A, Tropsha A. Beware of q2! <em>J Mol Graph Model</em>. 2002;20(4):269-276. <a href="https://doi.org/10.1016/S1093-3263(01)00123-1">doi:10.1016/S1093-3263(01)00123-1</a></li>
<li id="r11">Tropsha A. Best practices for QSAR model development, validation, and exploitation. <em>Mol Inform</em>. 2010;29(6-7):476-488. <a href="https://doi.org/10.1002/minf.201000061">doi:10.1002/minf.201000061</a></li>
<li id="r12">Gramatica P. Principles of QSAR models validation: internal and external. <em>QSAR Comb Sci</em>. 2007;26(5):694-701. <a href="https://doi.org/10.1002/qsar.200610151">doi:10.1002/qsar.200610151</a></li>
<li id="r13">Hawkins DM. The problem of overfitting. <em>J Chem Inf Comput Sci</em>. 2004;44(1):1-12. <a href="https://doi.org/10.1021/ci0342472">doi:10.1021/ci0342472</a></li>
<li id="r14">Peduzzi P, Concato J, Kemper E, Holford TR, Feinstein AR. A simulation study of the number of events per variable in logistic regression analysis. <em>J Clin Epidemiol</em>. 1996;49(12):1373-1379. <a href="https://doi.org/10.1016/S0895-4356(96)00236-3">doi:10.1016/S0895-4356(96)00236-3</a></li>
<li id="r15">Vittinghoff E, McCulloch CE. Relaxing the rule of ten events per variable in logistic and Cox regression. <em>Am J Epidemiol</em>. 2007;165(6):710-718. <a href="https://doi.org/10.1093/aje/kwk052">doi:10.1093/aje/kwk052</a></li>
<li id="r16">Albert A, Anderson JA. On the existence of maximum likelihood estimates in logistic regression models. <em>Biometrika</em>. 1984;71(1):1-10. <a href="https://doi.org/10.1093/biomet/71.1.1">doi:10.1093/biomet/71.1.1</a></li>
<li id="r17">Firth D. Bias reduction of maximum likelihood estimates. <em>Biometrika</em>. 1993;80(1):27-38. <a href="https://doi.org/10.1093/biomet/80.1.27">doi:10.1093/biomet/80.1.27</a></li>
<li id="r18">Van Calster B, McLernon DJ, van Smeden M, Wynants L, Steyerberg EW. Calibration: the Achilles heel of predictive analytics. <em>BMC Med</em>. 2019;17:230. <a href="https://doi.org/10.1186/s12916-019-1466-7">doi:10.1186/s12916-019-1466-7</a></li>
<li id="r19">Hoerl AE, Kennard RW. Ridge regression: biased estimation for nonorthogonal problems. <em>Technometrics</em>. 1970;12(1):55-67. <a href="https://doi.org/10.1080/00401706.1970.10488634">doi:10.1080/00401706.1970.10488634</a></li>
<li id="r20">Tibshirani R. Regression shrinkage and selection via the lasso. <em>J R Stat Soc B</em>. 1996;58(1):267-288. <a href="https://doi.org/10.1111/j.2517-6161.1996.tb02080.x">doi:10.1111/j.2517-6161.1996.tb02080.x</a></li>
<li id="r21">Zou H, Hastie T. Regularization and variable selection via the elastic net. <em>J R Stat Soc B</em>. 2005;67(2):301-320. <a href="https://doi.org/10.1111/j.1467-9868.2005.00503.x">doi:10.1111/j.1467-9868.2005.00503.x</a></li>
<li id="r22">Meinshausen N, Bühlmann P. Stability selection. <em>J R Stat Soc B</em>. 2010;72(4):417-473. <a href="https://doi.org/10.1111/j.1467-9868.2010.00740.x">doi:10.1111/j.1467-9868.2010.00740.x</a></li>
<li id="r23">Akaike H. A new look at the statistical model identification. <em>IEEE Trans Automat Contr</em>. 1974;19(6):716-723. <a href="https://doi.org/10.1109/TAC.1974.1100705">doi:10.1109/TAC.1974.1100705</a></li>
<li id="r24">Schwarz G. Estimating the dimension of a model. <em>Ann Stat</em>. 1978;6(2):461-464. <a href="https://doi.org/10.1214/aos/1176344136">doi:10.1214/aos/1176344136</a></li>
<li id="r25">Papapanou PN, Behle JH, Kebschull M, et al. Subgingival bacterial colonization profiles correlate with gingival tissue gene expression. <em>BMC Microbiol</em>. 2009;9:221. <a href="https://doi.org/10.1186/1471-2180-9-221">doi:10.1186/1471-2180-9-221</a></li>
<li id="r26">Breiman L. Statistical modeling: the two cultures. <em>Stat Sci</em>. 2001;16(3):199-231. <a href="https://doi.org/10.1214/ss/1009213726">doi:10.1214/ss/1009213726</a></li>
<li id="r27">Shmueli G. To explain or to predict? <em>Stat Sci</em>. 2010;25(3):289-310. <a href="https://doi.org/10.1214/10-STS330">doi:10.1214/10-STS330</a></li>
<li id="r28">Smyth GK. Linear models and empirical Bayes methods for assessing differential expression in microarray experiments. <em>Stat Appl Genet Mol Biol</em>. 2004;3:Article 3. <a href="https://doi.org/10.2202/1544-6115.1027">doi:10.2202/1544-6115.1027</a></li>
<li id="r29">Cramer RD, Patterson DE, Bunce JD. Comparative molecular field analysis (CoMFA). 1. Effect of shape on binding of steroids to carrier proteins. <em>J Am Chem Soc</em>. 1988;110(18):5959-5967. <a href="https://doi.org/10.1021/ja00226a005">doi:10.1021/ja00226a005</a></li>
<li id="r30">Krzywinski M, Altman N. Multiple linear regression. <em>Nat Methods</em>. 2015;12:1103-1104. <a href="https://doi.org/10.1038/nmeth.3665">doi:10.1038/nmeth.3665</a></li>
<li id="r31">Altman N, Krzywinski M. Association, correlation and causation. <em>Nat Methods</em>. 2015;12:899-900. <a href="https://doi.org/10.1038/nmeth.3587">doi:10.1038/nmeth.3587</a></li>
<li id="r32">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r33">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data - a statistical analysis. <em>PLoS ONE</em>. 2013;8(4):e61007. <a href="https://doi.org/10.1371/journal.pone.0061007">doi:10.1371/journal.pone.0061007</a></li>
<li id="r34">Fox J, Weisberg S. <em>An R Companion to Applied Regression</em>. 3rd ed. Sage; 2019. Software: the <code>car</code> package. <a href="https://doi.org/10.32614/CRAN.package.car">doi:10.32614/CRAN.package.car</a></li>
<li id="r35">Friedman J, Hastie T, Tibshirani R. Regularization paths for generalized linear models via coordinate descent. <em>J Stat Softw</em>. 2010;33(1):1-22. <a href="https://doi.org/10.18637/jss.v033.i01">doi:10.18637/jss.v033.i01</a></li>
<li id="r36">Bates D, Mächler M, Bolker B, Walker S. Fitting linear mixed-effects models using lme4. <em>J Stat Softw</em>. 2015;67(1):1-48. <a href="https://doi.org/10.18637/jss.v067.i01">doi:10.18637/jss.v067.i01</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 (free PDF). — Chapters 3, 4 and 6 are this unit at an introductory pace, with the same R functions.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009 (free PDF). — Chapters 3–4 for the geometry of ridge and lasso.</li>
<li>Gelman A, Hill J, Vehtari A. <em>Regression and Other Stories</em>. Cambridge University Press; 2020. — The best modern treatment of interpreting regression coefficients honestly; its examples are in R.</li>
<li>Harrell FE. <em>Regression Modeling Strategies</em>. 2nd ed. Springer; 2015. — Calibration, degrees-of-freedom budgeting and validation for clinical models; the companion <code>rms</code> package is R.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>The <code>glmnet</code> vignette "An Introduction to glmnet": <a href="https://glmnet.stanford.edu/articles/glmnet.html">glmnet.stanford.edu/articles/glmnet.html</a>.</li>
<li>The <code>car</code> package documentation on regression diagnostics: <a href="https://cran.r-project.org/package=car">cran.r-project.org/package=car</a>.</li>
<li>TeachOpenCADD talktorial on ligand-based screening and classical QSAR: <a href="https://projects.volkamerlab.org/teachopencadd/">projects.volkamerlab.org/teachopencadd</a>.</li>
<li>Practical Cheminformatics (P. Walters), posts on descriptor collinearity and on reading QSAR papers critically: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
</ul>
